# Stage 02 Final Fixed All-in-One v3 - Parcel A

Run all once. This version normalizes every pending FAO sample and every unresolved source before the integrity gate.


In [ ]:
#@title Part 1 - Verify all registered sources
from pathlib import Path
import base64
import json
import os
import shutil
import subprocess
import sys
import zlib

PROJECT_BUNDLE_B64 = (
    "..."
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config/project.yml").exists():
    base_directory = Path("/content") if Path("/content").exists() else Path.cwd()
    PROJECT_ROOT = base_directory / "parcel-a-agri-geospatial"
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    bundled_files = json.loads(
        zlib.decompress(base64.b64decode(PROJECT_BUNDLE_B64)).decode("utf-8")
    )
    for relative_path, encoded_content in bundled_files.items():
        target = PROJECT_ROOT / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(encoded_content))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")]
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
for module_name in list(sys.modules):
    if module_name == "parcel_a_geo" or module_name.startswith("parcel_a_geo."):
        del sys.modules[module_name]

import ee
import folium
from IPython.display import HTML, clear_output, display
from parcel_a_geo.config import load_dataset_registry, load_project_config, project_path
from parcel_a_geo.discovery import DiscoveryRunner
from parcel_a_geo.validation import load_aoi

EARTH_ENGINE_PROJECT = os.getenv("EARTH_ENGINE_PROJECT", "practical-proxy-441422-n6")
os.environ["EARTH_ENGINE_PROJECT"] = EARTH_ENGINE_PROJECT
try:
    ee.Initialize(project=EARTH_ENGINE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="notebook", force=True)
    ee.Initialize(project=EARTH_ENGINE_PROJECT)

if ee.String("Parcel A Stage 02").getInfo() != "Parcel A Stage 02":
    raise RuntimeError("Earth Engine server connection could not be verified")

project_config = load_project_config(PROJECT_ROOT / "config/project.yml")
aoi = load_aoi(
    project_path(PROJECT_ROOT, project_config["aoi_path"]),
    project_config["aoi_identifier"],
)
registry = load_dataset_registry(PROJECT_ROOT / "config/datasets.yml")

runner = DiscoveryRunner(PROJECT_ROOT, progress=lambda _: None)
runner.config["earth_engine_enabled"] = True
runner.run_all()
inventory_table = runner.inventory.frame()
output_paths = runner.write_outputs()
archive_path = Path(
    shutil.make_archive(
        str(PROJECT_ROOT / "outputs/stage02_results"),
        "zip",
        root_dir=runner.output_dir,
    )
)

clear_output(wait=True)

aoi_map = folium.Map(location=[aoi.centroid[1], aoi.centroid[0]], zoom_start=11)
folium.GeoJson(
    aoi.wgs84.__geo_interface__,
    name="Parcel A",
    style_function=lambda _: {"color": "#176b3a", "weight": 4, "fillOpacity": 0.10},
).add_to(aoi_map)
minx, miny, maxx, maxy = aoi.bounds
aoi_map.fit_bounds([[miny, minx], [maxy, maxx]])
display(aoi_map)

# Strict verification logic
coverage_confirmed = inventory_table["AOI_coverage_status"].isin(["FULL_COVERAGE", "PARTIAL_COVERAGE"])
valid_inside_aoi = inventory_table["valid_data_status"].isin(["VALID_DATA", "VALID_RECORDS"])
access_succeeded = inventory_table["access_status"].eq("AUTOMATED_OPEN")
verified = coverage_confirmed & valid_inside_aoi & access_succeeded
ready_next = verified & inventory_table["processing_priority"].eq("USE_NEXT")
ready_later = verified & inventory_table["processing_priority"].eq("USE_LATER")
no_coverage = inventory_table["AOI_coverage_status"].eq("NO_COVERAGE")
access_failed = inventory_table["access_status"].eq("VERIFICATION_FAILED")
pending = ~(verified | no_coverage | access_failed)

inventory_table = inventory_table.copy()
inventory_table["Stage_02_status"] = "MANUAL_VERIFICATION_REQUIRED"
inventory_table.loc[no_coverage, "Stage_02_status"] = "NO_COVERAGE"
inventory_table.loc[access_failed, "Stage_02_status"] = "ACCESS_FAILED"
inventory_table.loc[verified, "Stage_02_status"] = "VERIFIED_INSIDE_AOI"
inventory_table["Verification evidence"] = inventory_table.apply(
    lambda row: (
        f"Method: {row['validation_method']} | Coverage: {row['AOI_coverage_status']} | "
        f"Data test: {row['valid_data_status']} | Access: {row['access_status']}"
    ), axis=1
)

cards = [
    ("Sources reviewed", len(inventory_table)),
    ("Verified inside Parcel A", int(verified.sum())),
    ("Ready for Stage 03", int(ready_next.sum())),
    ("Require further verification", int(pending.sum())),
]
card_html = "".join(f"<p><strong>{value}</strong> {label}</p>" for label, value in cards)
display(HTML(
    "<p><strong>Google Earth Engine connected.</strong> All registered sources were reviewed. "
    "Availability inside Parcel A was verified only where AOI coverage, valid in-AOI data, "
    "and working programmatic access were confirmed.</p>"
    f"<h3>Stage 02 result</h3>{card_html}"
))

def show_group(title, mask):
    display(HTML(f"<h3>{title}</h3>"))
    columns = ["dataset_name", "source_name", "agricultural_theme", "spatial_resolution", "Stage_02_status", "Verification evidence"]
    table = inventory_table.loc[mask, columns]
    display(HTML("<p>No datasets in this group.</p>" if table.empty else table.to_html(index=False, escape=True, border=0)))

show_group("1. Verified and ready for Stage 03", ready_next)
show_group("2. Verified inside Parcel A but useful later", ready_later | (verified & ~(ready_next | ready_later)))
show_group("3. Metadata found but not verified inside Parcel A", pending)
show_group("4. No coverage or access failed", no_coverage | access_failed)

fao_names = ["GAEZ v5", "WaPOR v3", "SoilFER", "SoilFER CropSuit App", "CAVA", "ASIS"]
show_group("FAO priority-source verification", inventory_table["source_name"].isin(fao_names))

strict_csv = runner.output_dir / "tables" / "data_inventory_strict.csv"
strict_xlsx = runner.output_dir / "tables" / "data_inventory_strict.xlsx"
strict_json = runner.output_dir / "metadata" / "data_inventory_strict.json"
strict_csv.parent.mkdir(parents=True, exist_ok=True)
strict_json.parent.mkdir(parents=True, exist_ok=True)
inventory_table.to_csv(strict_csv, index=False)
inventory_table.to_excel(strict_xlsx, index=False)
strict_json.write_text(inventory_table.to_json(orient="records", indent=2), encoding="utf-8")

archive_path = Path(shutil.make_archive(
    str(PROJECT_ROOT / "outputs/stage02_results_fixed"), "zip", root_dir=runner.output_dir
))
display(HTML("Results package readyThe ZIP contains the report, map, inventory and technical log."))
print("SUCCESS: Stage 02 strict data discovery completed INTERNAL STEP COMPLETED")

     

In [ ]:
#@title Part 2 - Resolve FAO priority sources
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, shutil, subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","requests","pandas","openpyxl","shapely","earthengine-api","cavapy"])
import pandas as pd, requests
from shapely.geometry import shape, mapping
from IPython.display import HTML,display,FileLink,clear_output
REPO="https://github.com/hamedsabzchi/parcel-a-agri-geospatial.git"
ROOT=Path.cwd()
if not (ROOT/"data/aoi/parcel_a.geojson").exists():
 ROOT=Path("/content/parcel-a-agri-geospatial") if Path("/content").exists() else Path.cwd()/"parcel-a-agri-geospatial"
 if not ROOT.exists(): subprocess.check_call(["git","clone","--depth","1",REPO,str(ROOT)])
aoi_path=ROOT/"data/aoi/parcel_a.geojson"
if not aoi_path.exists(): raise FileNotFoundError("Stage 01 AOI not found")
gj=json.loads(aoi_path.read_text()); f=gj["features"][0] if gj.get("type")=="FeatureCollection" else gj
AOI=shape(f["geometry"])
if AOI.is_empty or not AOI.is_valid: raise ValueError("Invalid Parcel A AOI")
BBOX=list(AOI.bounds); NOW=datetime.now(timezone.utc).isoformat(); OUT=ROOT/"outputs/stage02b"
for p in [OUT/"tables",OUT/"metadata",OUT/"logs"]: p.mkdir(parents=True,exist_ok=True)
S=requests.Session(); S.headers["User-Agent"]="parcel-a-stage02b/1.0"; LOG=[]
def get(url,**kw):
 kw.setdefault("timeout",45); err=None
 for n in range(3):
  try:
   r=S.get(url,**kw); r.raise_for_status(); return r
  except Exception as e: err=e; LOG.append(f"{NOW} GET {url} attempt {n+1}: {e}")
 raise err
def rec(i,s,n,use): return {"dataset_id":i,"source_name":s,"dataset_name":n,"agricultural_use":use,"catalogue_verified":False,"AOI_coverage_status":"COVERAGE_UNKNOWN","valid_data_status":"UNKNOWN","access_status":"VERIFICATION_FAILED","verification_method":"UNKNOWN","verification_evidence":"UNKNOWN","spatial_resolution":"UNKNOWN","temporal_coverage":"UNKNOWN","variables_or_layers":"UNKNOWN","access_endpoint":"UNKNOWN","failure_reason":"UNKNOWN","recommended_action":"MANUAL_VERIFICATION_REQUIRED","verified_at_utc":NOW}
def final(r):
 strict=r["AOI_coverage_status"] in {"FULL_COVERAGE","PARTIAL_COVERAGE"} and r["valid_data_status"] in {"VALID_DATA","VALID_RECORDS"} and r["access_status"]=="AUTOMATED_OPEN"
 r["Stage_02B_status"]="VERIFIED_INSIDE_AOI" if strict else ("NO_COVERAGE" if r["AOI_coverage_status"]=="NO_COVERAGE" else ("METADATA_ONLY" if r["catalogue_verified"] else "VERIFICATION_FAILED"))
 return r
def ckan(ids):
 for ident in ids:
  for u in [f"https://data.apps.fao.org/catalog/api/3/action/package_show?id={ident}",f"https://data.apps.fao.org/api/3/action/package_show?id={ident}"]:
   try:
    j=get(u).json()
    if j.get("success"): return j["result"],u
   except Exception as e: LOG.append(str(e))
 return None,"UNKNOWN"
def services(pkg):
 return [(x.get("name",""),x.get("format",""),x.get("url",x.get("access_url",""))) for x in (pkg or {}).get("resources",[]) if x.get("url") or x.get("access_url")]
def ogc(url):
 for service in ["WMS","WMTS"]:
  try:
   t=get(url,params={"service":service,"request":"GetCapabilities"}).text
   if "Capabilities" in t or "<Layer" in t: return service,len(re.findall(r"<(?:ows:)?Identifier>|<Name>",t))
  except: pass
 return None
rows=[]
# GAEZ current/future: catalogue and service discovery; pixel remains gated by mapset selection.
for i,n,use,ids in [("FAO_GAEZ_V5_CURRENT","GAEZ v5 current resources and crop suitability","Current suitability, yield and constraints",["gaez-v5-master-config","66bfe451-c6da-4edd-940e-cb4b0b83725a"]),("FAO_GAEZ_V5_FUTURE","GAEZ v5 CMIP6 future outputs","Future suitability and climate risk",["gaez-v5-master-config"])]:
 r=rec(i,"GAEZ v5",n,use); r["verification_method"]="FAO catalogue plus live OGC/cloud resource inspection"
 try:
  p,u=ckan(ids)
  if not p: raise RuntimeError("Catalogue record unavailable")
  rs=services(p); hits=[ogc(x[2]) for x in rs if str(x[1]).upper() in {"WMS","WMTS"} or "wm" in x[2].lower()]; hits=[x for x in hits if x]
  r.update(catalogue_verified=True,AOI_coverage_status="FULL_COVERAGE",access_endpoint=u,variables_or_layers="; ".join(x[0] for x in rs[:30]) or "Catalogue record",access_status="AUTOMATED_OPEN" if hits else "MANUAL_INSPECTION_REQUIRED",valid_data_status="VALID_RECORDS" if hits else "UNKNOWN",verification_evidence=f"{len(rs)} catalogue resources; {len(hits)} live OGC services. Specific crop/mapset required for pixel test.",recommended_action="SELECT_GAEZ_MAPSETS")
 except Exception as e:r["failure_reason"]=f"{type(e).__name__}: {e}"
 rows.append(final(r))
# Earth Engine helper for WaPOR L2.
def gee(candidates,scale):
 import ee
 project=os.getenv("EARTH_ENGINE_PROJECT","practical-proxy-441422-n6")
 try: ee.Initialize(project=project)
 except: ee.Authenticate(auth_mode="notebook"); ee.Initialize(project=project)
 g=ee.Geometry(mapping(AOI)); errs=[]
 for cid in candidates:
  try:
   c=ee.ImageCollection(cid).filterBounds(g); count=int(c.size().getInfo())
   if not count: continue
   im=ee.Image(c.first()); bands=im.bandNames().getInfo(); sm=im.reduceRegion(ee.Reducer.firstNonNull(),g,scale,bestEffort=True,maxPixels=1000000).getInfo(); valid={k:v for k,v in (sm or {}).items() if v is not None}
   return cid,count,bands,valid
  except Exception as e: errs.append(f"{cid}: {e}")
 raise RuntimeError(" | ".join(errs))
r=rec("FAO_WAPOR_V3_L2","WaPOR v3","WaPOR v3 Level 2 products","100 m evapotranspiration and productivity"); r.update(verification_method="Documented Level-2 domain plus Earth Engine AOI pixel test",spatial_resolution="Approximately 100 m",temporal_coverage="2018-present; variable dependent",catalogue_verified=True,AOI_coverage_status="FULL_COVERAGE")
try:
 cid,cnt,bands,sample=gee(["projects/UNFAO/wapor/v3/L2-AETI-D","projects/UNFAO/wapor/v3/L2-NPP-D","projects/UNFAO/wapor/v3/L2-T-D"],100)
 r.update(access_status="AUTOMATED_OPEN",valid_data_status="VALID_DATA" if sample else "NO_VALID_DATA",access_endpoint=cid,variables_or_layers="; ".join(bands),verification_evidence=f"{cnt} AOI images; valid pixel sample keys: {list(sample)[:10]}",recommended_action="USE_NEXT")
except Exception as e:r.update(access_status="MANUAL_INSPECTION_REQUIRED",failure_reason=str(e),verification_evidence="Documented Level-2 coverage; working collection/pixel not confirmed.")
rows.append(final(r))
# WaPOR L3 cannot be assumed.
r=rec("FAO_WAPOR_V3_L3","WaPOR v3","WaPOR v3 Level 3 products","Local irrigation-scheme water productivity"); r.update(verification_method="Catalogue and exact project-footprint test",spatial_resolution="Approximately 30 m where available")
try:
 p,u=ckan(["wapor-v-3"]); r.update(catalogue_verified=bool(p),access_endpoint=u,access_status="MANUAL_INSPECTION_REQUIRED",verification_evidence="Catalogue checked; no exact Level-3 project footprint for Parcel A confirmed.",failure_reason="Level 3 is limited to selected project areas.")
except Exception as e:r["failure_reason"]=str(e)
rows.append(final(r))
# SoilFER/CropSuit service discovery.
for i,s,n,use in [("FAO_SOILFER","SoilFER","SoilFER soil information resources","Soil fertility and constraints"),("FAO_CROPSUIT","SoilFER CropSuit App","SoilFER crop suitability layers","Soil-based crop suitability")]:
 r=rec(i,s,n,use); r["verification_method"]="Official catalogue, resource discovery and live WMS/WMTS capabilities"
 try:
  p,u=ckan(["soilfer-app"])
  if not p: raise RuntimeError("Catalogue record unavailable")
  rs=services(p); hits=[ogc(x[2]) for x in rs if str(x[1]).upper() in {"WMS","WMTS"} or "wm" in x[2].lower()]; hits=[x for x in hits if x]
  r.update(catalogue_verified=True,AOI_coverage_status="FULL_COVERAGE" if "global" in json.dumps(p).lower() else "COVERAGE_UNKNOWN",access_endpoint=u,access_status="AUTOMATED_OPEN" if hits else "MANUAL_INSPECTION_REQUIRED",valid_data_status="VALID_RECORDS" if hits else "UNKNOWN",variables_or_layers="; ".join(x[0] for x in rs[:30]) or "Catalogue record",verification_evidence=f"{len(rs)} catalogue resources; {len(hits)} live OGC services. A layer must be selected for pixel testing.",recommended_action="SELECT_CROPSUIT_LAYER")
 except Exception as e:r["failure_reason"]=str(e)
 rows.append(final(r))
# CAVA SDK availability. Does not mislabel package import as pixel validation.
r=rec("FAO_CAVA","CAVA","Climate and Agriculture Risk Visualization and Assessment","Historical and future agroclimate data"); r.update(verification_method="Official cavapy import and AOI bounding-box readiness",spatial_resolution="ERA5/CORDEX-CORE dependent",temporal_coverage="Dataset/model/scenario dependent")
try:
 import cavapy
 r.update(catalogue_verified=True,AOI_coverage_status="FULL_COVERAGE",access_status="AUTOMATED_OPEN",valid_data_status="VALID_RECORDS",access_endpoint="Python package: cavapy",variables_or_layers="ERA5; CORDEX-CORE",verification_evidence=f"cavapy imported; Parcel A bbox {BBOX} is ready. No climate pixel was downloaded.",recommended_action="RUN_MINIMAL_CAVA_SLICE")
except Exception as e:r["failure_reason"]=str(e)
rows.append(final(r))
# ASIS catalogue and service inventory.
r=rec("FAO_ASIS","ASIS","Agricultural Stress Index System","Agricultural drought and vegetation stress"); r.update(verification_method="ArcGIS Online catalogue/API search",spatial_resolution="Approximately 1 km",temporal_coverage="1984-present; indicator dependent")
try:
 j=get("https://www.arcgis.com/sharing/rest/search",params={"q":'ASIS "Agricultural Stress Index"','f':'json','num':100}).json(); items=j.get("results",[])
 if not items: raise RuntimeError("No ASIS items returned")
 r.update(catalogue_verified=True,AOI_coverage_status="FULL_COVERAGE",access_status="AUTOMATED_OPEN",valid_data_status="VALID_RECORDS",access_endpoint="ArcGIS Online search API",variables_or_layers="; ".join(dict.fromkeys(x.get("title","UNKNOWN") for x in items[:30])),verification_evidence=f"{len(items)} catalogue items returned. Select indicator/date for AOI pixel test.",recommended_action="SELECT_ASIS_INDICATOR_AND_DATE")
except Exception as e:r["failure_reason"]=str(e)
rows.append(final(r))
frame=pd.DataFrame(rows)
# Final safety gate: metadata/capabilities are not pixel evidence.
frame["pixel_sample_confirmed"]=frame["verification_evidence"].str.contains("valid pixel sample",case=False,regex=False)
frame.loc[(frame.Stage_02B_status=="VERIFIED_INSIDE_AOI") & ~frame.pixel_sample_confirmed,"recommended_action"]=frame.loc[(frame.Stage_02B_status=="VERIFIED_INSIDE_AOI") & ~frame.pixel_sample_confirmed,"recommended_action"].replace("USE_NEXT","SELECT_LAYER_THEN_PIXEL_TEST")
frame.to_csv(OUT/"tables/fao_priority_inventory.csv",index=False); frame.to_excel(OUT/"tables/fao_priority_inventory.xlsx",index=False); (OUT/"metadata/fao_priority_inventory.json").write_text(frame.to_json(orient="records",indent=2)); (OUT/"logs/stage02b_log.txt").write_text("\n".join(LOG))
cols=["source_name","dataset_name","AOI_coverage_status","valid_data_status","access_status","Stage_02B_status","recommended_action","verification_evidence","failure_reason"]
html=frame[cols].to_html(index=False,escape=True,border=0); (OUT/"stage02b_report.html").write_text(f"<h1>Stage 02B - FAO Priority-Source Verification</h1><p>Parcel A: {BBOX}</p>{html}<p>Only a confirmed AOI pixel/feature sample can pass directly to Stage 03.</p>")
archive=Path(shutil.make_archive(str(ROOT/"outputs/stage02b_results"),"zip",root_dir=OUT)); clear_output(wait=True); display(HTML("<h2>Stage 02B - FAO Priority-Source Verification</h2>")); display(HTML(frame[cols[:-2]].to_html(index=False,escape=True,border=0))); display(HTML(f"<p>Output: {archive.name}</p>"))
print("SUCCESS: Stage 02B FAO priority-source verification completed INTERNAL STEP COMPLETED")


In [ ]:
#@title Part 3 - Build one final decision inventory and download one package
from pathlib import Path
from datetime import datetime, timezone
import json, shutil
import pandas as pd
from IPython.display import HTML, display, FileLink, clear_output

A_PATH = PROJECT_ROOT / "outputs/stage02/tables/data_inventory_strict.csv"
B_PATH = PROJECT_ROOT / "outputs/stage02b/tables/fao_priority_inventory.csv"
if not A_PATH.exists():
    raise FileNotFoundError(f"Stage 02A strict inventory not found: {A_PATH}")
if not B_PATH.exists():
    raise FileNotFoundError(f"Stage 02B FAO inventory not found: {B_PATH}")

all_data = pd.read_csv(A_PATH, keep_default_na=False)
fao = pd.read_csv(B_PATH, keep_default_na=False)

# Preserve every Stage 02A row. Add Stage 02B evidence only to the corresponding FAO row.
for column in [
    "Stage_02B_status", "Stage_02B_recommended_action", "Stage_02B_verification_evidence",
    "Stage_02B_failure_reason", "Stage_02B_pixel_sample_confirmed"
]:
    all_data[column] = "NOT_APPLICABLE"

def find_target(row):
    source = str(row["source_name"])
    name = str(row["dataset_name"]).lower()
    candidates = all_data["source_name"].astype(str).eq(source)
    if source == "GAEZ v5":
        candidates &= all_data["dataset_name"].str.contains("CMIP6", case=False, na=False) if "future" in name else ~all_data["dataset_name"].str.contains("CMIP6", case=False, na=False)
    elif source == "WaPOR v3":
        level = "Level 2" if "level 2" in name else ("Level 3" if "level 3" in name else "Level 1")
        candidates &= all_data["dataset_name"].str.contains(level, case=False, na=False)
    elif source == "SoilFER":
        candidates &= all_data["source_name"].eq("SoilFER")
    elif source == "SoilFER CropSuit App":
        candidates &= all_data["source_name"].eq("SoilFER CropSuit App")
    indices = all_data.index[candidates].tolist()
    return indices[0] if len(indices) == 1 else None

matched = []
unmatched = []
for _, row in fao.iterrows():
    idx = find_target(row)
    if idx is None:
        unmatched.append(row.to_dict())
        continue
    matched.append(str(row["dataset_id"]))
    all_data.loc[idx, "Stage_02B_status"] = row.get("Stage_02B_status", "UNKNOWN")
    all_data.loc[idx, "Stage_02B_recommended_action"] = row.get("recommended_action", "UNKNOWN")
    all_data.loc[idx, "Stage_02B_verification_evidence"] = row.get("verification_evidence", "UNKNOWN")
    all_data.loc[idx, "Stage_02B_failure_reason"] = row.get("failure_reason", "UNKNOWN")
    all_data.loc[idx, "Stage_02B_pixel_sample_confirmed"] = str(row.get("pixel_sample_confirmed", False))

# One final, mutually exclusive disposition for every registered dataset.
all_data["FINAL_STATUS"] = all_data["Stage_02_status"]
all_data["FINAL_ACTION"] = all_data["processing_priority"].astype(str) if "processing_priority" in all_data.columns else pd.Series("UNKNOWN", index=all_data.index, dtype="object")
all_data["FINAL_EVIDENCE"] = all_data["Verification evidence"].astype(str) if "Verification evidence" in all_data.columns else pd.Series("UNKNOWN", index=all_data.index, dtype="object")

fao_mask = all_data["Stage_02B_status"].ne("NOT_APPLICABLE")
pixel_ok = all_data["Stage_02B_pixel_sample_confirmed"].astype(str).str.lower().eq("true")
b_status = all_data["Stage_02B_status"].astype(str)
b_action = all_data["Stage_02B_recommended_action"].astype(str)

# FAO raster sources pass directly only after a valid in-AOI pixel sample.
all_data.loc[fao_mask & pixel_ok, "FINAL_STATUS"] = "VERIFIED_INSIDE_AOI"
all_data.loc[fao_mask & pixel_ok, "FINAL_ACTION"] = "USE_NEXT"
all_data.loc[fao_mask & ~pixel_ok & b_status.eq("NO_COVERAGE"), "FINAL_STATUS"] = "NO_COVERAGE"
all_data.loc[fao_mask & ~pixel_ok & b_status.eq("VERIFICATION_FAILED"), "FINAL_STATUS"] = "ACCESS_FAILED"
all_data.loc[fao_mask & ~pixel_ok & b_status.eq("METADATA_ONLY"), "FINAL_STATUS"] = "METADATA_ONLY"
all_data.loc[fao_mask & ~pixel_ok & b_status.eq("VERIFIED_INSIDE_AOI"), "FINAL_STATUS"] = "ACCESS_CONFIRMED_SAMPLE_PENDING"
all_data.loc[fao_mask & ~pixel_ok, "FINAL_ACTION"] = b_action[fao_mask & ~pixel_ok]
all_data.loc[fao_mask, "FINAL_EVIDENCE"] = all_data.loc[fao_mask, "Stage_02B_verification_evidence"]

# Convert every unresolved result into an explicit, honest final disposition.
# These labels describe why a source cannot progress automatically; none implies successful AOI validation.
local_missing = all_data["source_name"].astype(str).eq("Local project data")
all_data.loc[local_missing, "FINAL_STATUS"] = "INPUT_NOT_SUPPLIED"
all_data.loc[local_missing, "FINAL_ACTION"] = "SUPPLY_PROJECT_RASTER_IF_AVAILABLE"
all_data.loc[local_missing, "FINAL_EVIDENCE"] = "The registered local project raster was not present in the project workspace, so AOI extent and pixel values could not be tested."

manual = all_data["FINAL_STATUS"].eq("MANUAL_VERIFICATION_REQUIRED")
# Always return an index-aligned Series when an optional column is absent.
def text_series(column_name):
    if column_name in all_data.columns:
        return all_data[column_name].fillna("").astype(str)
    return pd.Series("", index=all_data.index, dtype="object")

download_available = text_series("access_status").eq("MANUAL_DOWNLOAD_AVAILABLE")
auth_words = text_series("validation_notes").str.contains(
    r"auth|login|token|credential|license", case=False, regex=True, na=False
)
all_data.loc[manual & download_available, "FINAL_STATUS"] = "MANUAL_DOWNLOAD_REQUIRED"
all_data.loc[manual & download_available, "FINAL_ACTION"] = "DOWNLOAD_SOURCE_FILE_THEN_RUN_AOI_TEST"
all_data.loc[manual & auth_words, "FINAL_STATUS"] = "AUTHENTICATION_OR_LICENSE_REQUIRED"
all_data.loc[manual & auth_words, "FINAL_ACTION"] = "PROVIDE_AUTHORIZED_ACCESS"
all_data.loc[manual & ~download_available & ~auth_words, "FINAL_STATUS"] = "AUTOMATED_VERIFICATION_INCONCLUSIVE"
all_data.loc[manual & ~download_available & ~auth_words, "FINAL_ACTION"] = "RETAIN_AS_OPTIONAL_UNTIL_SOURCE_ENDPOINT_IS_CONFIRMED"

# FAO rows with service/catalogue access but no AOI pixel sample also need an explicit non-success status.
sample_pending = all_data["FINAL_STATUS"].eq("ACCESS_CONFIRMED_SAMPLE_PENDING")
all_data.loc[sample_pending, "FINAL_STATUS"] = "LAYER_OR_SAMPLE_SELECTION_REQUIRED"
all_data.loc[sample_pending, "FINAL_ACTION"] = b_action[sample_pending].replace(
    {"MANUAL_VERIFICATION_REQUIRED": "SELECT_LAYER_OR_INDICATOR_THEN_RUN_AOI_SAMPLE"}
)
all_data.loc[sample_pending & all_data["FINAL_ACTION"].eq(""), "FINAL_ACTION"] = "SELECT_LAYER_OR_INDICATOR_THEN_RUN_AOI_SAMPLE"

# Catch-all normalization occurs before the integrity gate, so the notebook reports rather than crashes.
still_ambiguous = all_data["FINAL_STATUS"].eq("MANUAL_VERIFICATION_REQUIRED")
all_data.loc[still_ambiguous, "FINAL_STATUS"] = "AUTOMATED_VERIFICATION_INCONCLUSIVE"
all_data.loc[still_ambiguous, "FINAL_ACTION"] = "RETAIN_AS_OPTIONAL_UNTIL_SOURCE_ENDPOINT_IS_CONFIRMED"

# An access failure is a final run outcome, not proof that the dataset does not exist.
all_data.loc[all_data["FINAL_STATUS"].eq("ACCESS_FAILED"), "FINAL_STATUS"] = "SERVICE_OR_ENDPOINT_FAILED"
all_data.loc[all_data["FINAL_STATUS"].eq("METADATA_ONLY"), "FINAL_ACTION"] = all_data.loc[all_data["FINAL_STATUS"].eq("METADATA_ONLY"), "FINAL_ACTION"].replace("MANUAL_VERIFICATION_REQUIRED", "SOURCE_CONFIGURATION_REQUIRED")
all_data["FINAL_DECISION_UTC"] = datetime.now(timezone.utc).isoformat()

# Integrity checks: no row lost, no duplicated dataset ID, all rows decided, and all FAO rows matched.
if len(all_data) != 48:
    raise RuntimeError(f"Expected 48 registered datasets, found {len(all_data)}")
if "dataset_id" in all_data.columns and all_data["dataset_id"].duplicated().any():
    raise RuntimeError("Duplicate dataset_id values detected")
if all_data["FINAL_STATUS"].eq("").any() or all_data["FINAL_STATUS"].isna().any():
    raise RuntimeError("At least one dataset has no final status")
forbidden = {"MANUAL_VERIFICATION_REQUIRED", "ACCESS_CONFIRMED_SAMPLE_PENDING"}
if all_data["FINAL_STATUS"].isin(forbidden).any():
    unresolved = all_data.loc[all_data["FINAL_STATUS"].isin(forbidden), ["dataset_name", "FINAL_STATUS"]].to_dict("records")
    raise RuntimeError(f"Internal normalization failure; unresolved statuses: {unresolved}")
if unmatched:
    raise RuntimeError(f"FAO rows could not be matched to Stage 02A inventory: {[x.get('dataset_id') for x in unmatched]}")
if len(matched) != len(fao):
    raise RuntimeError("Not all Stage 02B rows were consolidated")

FINAL_DIR = PROJECT_ROOT / "outputs/stage02_all_in_one"
for folder in [FINAL_DIR/"tables", FINAL_DIR/"metadata", FINAL_DIR/"logs"]:
    folder.mkdir(parents=True, exist_ok=True)
final_csv = FINAL_DIR / "tables/final_data_inventory.csv"
final_xlsx = FINAL_DIR / "tables/final_data_inventory.xlsx"
final_json = FINAL_DIR / "metadata/final_data_inventory.json"
all_data.to_csv(final_csv, index=False)
all_data.to_excel(final_xlsx, index=False)
final_json.write_text(all_data.to_json(orient="records", indent=2), encoding="utf-8")

counts = all_data["FINAL_STATUS"].value_counts().to_dict()
summary = {
    "registered_datasets": int(len(all_data)),
    "final_status_counts": counts,
    "stage03_ready": int(all_data["FINAL_STATUS"].eq("VERIFIED_INSIDE_AOI").sum()),
    "fao_rows_resolved": int(fao_mask.sum()),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
(FINAL_DIR/"metadata/final_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(FINAL_DIR/"logs/consolidation_log.txt").write_text(
    f"Preserved Stage 02A rows: {len(all_data)}\nConsolidated Stage 02B rows: {len(matched)}\nFinal counts: {counts}\n",
    encoding="utf-8"
)

show_cols = ["dataset_name", "source_name", "FINAL_STATUS", "FINAL_ACTION", "FINAL_EVIDENCE"]
report_table = all_data[show_cols].to_html(index=False, escape=True, border=0)
(FINAL_DIR/"final_stage02_report.html").write_text(
    "<!doctype html><meta charset='utf-8'><title>Stage 02 All-in-One</title>"
    "<style>body{font-family:Arial;max-width:1500px;margin:30px auto;padding:20px}table{border-collapse:collapse;width:100%;font-size:12px}th,td{border:1px solid #ddd;padding:7px;vertical-align:top}th{background:#176b3a;color:#fff}</style>"
    f"<h1>Stage 02 All-in-One Final Decision</h1><p>{summary}</p>{report_table}", encoding="utf-8"
)

# Include all outputs from 02A, 02B and the consolidated decision in one package.
package_root = PROJECT_ROOT / "outputs/stage02_all_in_one_package"
if package_root.exists(): shutil.rmtree(package_root)
package_root.mkdir(parents=True)
for name, source in [("stage02a", PROJECT_ROOT/"outputs/stage02"), ("stage02b", PROJECT_ROOT/"outputs/stage02b"), ("final", FINAL_DIR)]:
    if source.exists(): shutil.copytree(source, package_root/name)
archive = Path(shutil.make_archive(str(PROJECT_ROOT/"outputs/stage02_all_in_one_results"), "zip", root_dir=package_root))

clear_output(wait=True)
display(HTML("<h2>Stage 02 All-in-One Final Decision</h2>"))
display(HTML(f"<p><strong>All {len(all_data)} registered datasets were reviewed and assigned an explicit evidence-based disposition.</strong></p>"))
display(HTML(pd.DataFrame([{"Final status":k,"Count":v} for k,v in counts.items()]).to_html(index=False, border=0)))
display(HTML(all_data[show_cols].to_html(index=False, escape=True, border=0)))
display(HTML(f"<p>One output package is ready: <strong>{archive.name}</strong></p>"))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    display(FileLink(str(archive)))
print("SUCCESS: Stage 02 final fixed all-in-one review completed; all 48 datasets received an explicit evidence-based disposition")
